# Slow-oscillation scan: results

Results of the Optuna scans of `optuna_scan.py` (network of `baseline.py`).

- **hit**: `L5Exc` oscillates with > 8 mV peak-to-peak in 3-7 Hz, rates below `RATE_MAX`, V above `V_FLOOR`
- `SAMPLER = 'tpe'`: adaptive search (concentrated around the hits, biased probabilities)
- `SAMPLER = 'qmc'`: uniform Sobol sampling (unbiased map of the space, merged from the per-worker studies)

Parameters are shown relative to their baseline value (x1 = `baseline.py`).

In [ ]:
import sys, os, pathlib
HERE = pathlib.Path.cwd()
if not (HERE / 'optuna_scan.py').exists():  # notebook started from the repo root
    HERE = HERE / 'oscill'
sys.path.insert(0, str(HERE))
os.chdir(HERE)

import numpy as np
import optuna
import plotly.graph_objects as go
import matplotlib.pylab as plt
import brian2 as b2
b2.BrianLogger.suppress_name('brian2')
optuna.logging.set_verbosity(optuna.logging.WARNING)

import optuna_scan as scan

## Choose the study

In [ ]:
SAMPLER = 'tpe'   # 'tpe' or 'qmc'
name = scan.study_name(SAMPLER)
T = scan.load_table(name)
scan.show(name)

## Summary

A: outcome of the trials, B: amplitude vs frequency (physiological trials),
C: hit probability along each parameter, D: hit probability for the two most influential parameters.

In [ ]:
scan.summary(name)

## Hits vs misses, interactively

Hover a point to get its trial id (to plot its traces below), click the legend to hide categories.

In [ ]:
COLORS = {'runaway': 'orange', 'rate': 'purple', 'vfloor': 'saddlebrown',
          'band': 'steelblue', 'small': 'lightgrey', 'hit': 'red'}
pop = scan.TARGET_POPS[0]
rng = np.random.default_rng(0)
fig = go.Figure()
for c, label in scan.CATEGORIES.items():
    idx = np.flatnonzero(T['category'] == c)
    if c != 'hit':
        idx = idx[rng.permutation(len(idx))[:5000]]  # subsample the misses
    fig.add_trace(go.Scattergl(
        x=T['freq_%s' % pop][idx], y=T['amp_%s' % pop][idx], mode='markers',
        name='%s (%i)' % (label, (T['category'] == c).sum()),
        marker=dict(size=4 if c == 'hit' else 3, color=COLORS[c]),
        text=['%s<br>rate %.1f Hz, Vlow %.0f mV' % (T['id'][i], T['rate_%s' % pop][i],
                                                    T['vlow_%s' % pop][i]) for i in idx],
        hovertemplate='%{text}<br>%{x} Hz, %{y:.1f} mV<extra></extra>'))
fig.add_vrect(x0=scan.BAND[0], x1=scan.BAND[1], fillcolor='red', opacity=0.08, line_width=0)
fig.add_hline(y=scan.AMP_MIN, line_dash='dot', line_color='red')
fig.update_layout(xaxis_title='%s peak frequency (Hz)' % pop,
                  yaxis_title='%s 3-7 Hz amplitude (mV pk-pk)' % pop,
                  height=500, template='plotly_white')
fig.show()

## The hit region: parallel coordinates

One line per hit, axes = parameters (log2 of value / baseline: 0 = baseline, ±2 = x4 / x1/4),
color = `L5Exc` amplitude. Drag along an axis to select a range and see how the other parameters follow.

In [ ]:
hits = np.flatnonzero(T['hit'])
bins = np.linspace(-np.log(scan.RANGE), np.log(scan.RANGE), 13)
P = np.array([scan._hit_probability(T, k, bins) for k in range(len(T['names']))])
order = np.argsort(-(np.nanmax(P, 1) - np.nanmin(P, 1)))  # most influential first
r = np.log2(scan.RANGE)
fig = go.Figure(go.Parcoords(
    line=dict(color=T['amp_%s' % pop][hits], colorscale='Viridis', showscale=True,
              colorbar=dict(title='amp (mV)')),
    dimensions=[dict(label=T['names'][k].replace('indegree', 'K').replace('_', ' '),
                     values=np.log2(T['X'][hits, k]), range=[-r, r])
                for k in order]))
fig.update_layout(height=550, margin=dict(l=80, r=80, t=80, b=40))
fig.show()

## Projections of the score on the 2D subspaces

Lower triangle: mean score of the trials in each bin of a parameter pair; diagonal: mean score along each parameter (± s.e.m.).
The score is the objective function: `L5Exc` 3-7 Hz amplitude capped at 8 mV, set to 0 for infeasible trials (`feasible_only=True`),
so a bin at 8 means all its trials are hits. White: no trial in the bin. `+`: baseline.

`PARAMS`: the parameters to show (default: the `n` most influential), e.g. `['w_L5Inh_L5Exc', 'indegree_ReInh_ReInh', ...]`.

In [ ]:
PARAMS = None   # or a list of parameter names (see T['names'])
fig, AX = scan.projections(name, params=PARAMS, n=8, nbins=10, feasible_only=True, T=T)

### One subspace, trial by trial

All trials in the plane of two parameters, colored by score (infeasible trials as small grey points); hover for the trial id.

In [ ]:
P1, P2 = T['names'][order[0]], T['names'][order[1]]   # or any two names of T['names']

k1, k2 = T['names'].index(P1), T['names'].index(P2)
score = np.where(T['feasible'], T['score'], 0)
fig = go.Figure()
for label, ok, kw in [('infeasible', ~T['feasible'], dict(size=3, color='lightgrey')),
                      ('feasible', T['feasible'], dict(size=5, color=score[T['feasible']], colorscale='Viridis',
                                    cmin=0, cmax=scan.AMP_MIN, showscale=True,
                                    colorbar=dict(title='score (mV)')))]:
    idx = np.flatnonzero(ok)
    fig.add_trace(go.Scattergl(
        x=T['X'][idx, k1], y=T['X'][idx, k2], mode='markers', marker=kw,
        name=label,
        text=['%s, score %.1f' % (T['id'][i], score[i]) for i in idx],
        hovertemplate='%{text}<extra></extra>'))
fig.add_trace(go.Scatter(x=[1], y=[1], mode='markers', name='baseline',
                         marker=dict(symbol='cross', size=12, color='black')))
fig.update_xaxes(type='log', title=P1 + ' (x baseline)')
fig.update_yaxes(type='log', title=P2 + ' (x baseline)')
fig.update_layout(height=600, width=700, template='plotly_white')
fig.show()

## Optuna's built-in plots

They show the *score* (amplitude capped at 8 mV), infeasible trials (constraints violated) in grey.
For the qmc study, the per-worker studies are merged into one (subsampled) in-memory study.

In [ ]:
def as_study(name, max_trials=20000, seed=0):
    shards = scan.shards(name)
    if len(shards) == 1:
        return shards[0][1]
    trials = [t for _, s in shards for t in s.get_trials(
        deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])]
    keep = np.random.default_rng(seed).permutation(len(trials))[:max_trials]
    study = optuna.create_study(direction='maximize')
    study.add_trials([trials[i] for i in keep])
    return study

study = as_study(name)
top = [T['names'][k] for k in order[:4]]
print('most influential parameters:', top)

In [ ]:
optuna.visualization.plot_optimization_history(study).show()

In [ ]:
optuna.visualization.plot_slice(study, params=top).show()

In [ ]:
optuna.visualization.plot_contour(study, params=top[:2]).show()

## Traces of a trial

Re-runs the simulation (~20 s). `TRIAL`: an id from the hover above (e.g. `'17756'` for tpe, `'w03:120'` for qmc),
or the largest hit by default.

In [ ]:
TRIAL = scan.best_hits(name, 1)[0]
print('trial', TRIAL, dict(scan.get_trial(name, TRIAL).params))
scan.plot_trial(name, TRIAL)